# Session1_Task4 — Product Performance Analysis

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.backends.backend_pdf import PdfPages

s = pd.read_csv('sales_transactions_cleaned.csv')
p = pd.read_csv('products.csv')

s['revenue'] = (s['quantity'] * s['price']) - pd.to_numeric(s['discount_amount'], errors='coerce').fillna(0)

In [2]:
# --- Clean products.csv ---

# r'[^0-9.\-]' → Regex ลบทุกอักขระที่ไม่ใช่ตัวเลข, จุด, หรือ - (เช่น ลบ $ ออก)
# .abs()        → แปลงค่าติดลบให้เป็นบวก
def clean_num(col):
    return pd.to_numeric(
        col.astype(str).str.replace(r'[^0-9.\-]', '', regex=True),
        errors='coerce'
    ).abs()

p['price_c'] = clean_num(p['price'])
p['cost_c']  = clean_num(p['cost'])

# .replace() → แก้ category ที่สะกดผิด
p['category'] = p['category'].replace({'Pastry': 'Pastries'})

# profit margin ต่อชิ้น = ราคาขาย - ต้นทุน
p['profit_margin'] = p['price_c'] - p['cost_c']

In [3]:
# --- Merge + คำนวณ ---

# .merge(how='left') → JOIN เก็บทุกแถวของ sales ไว้
tq = (s.assign(revenue=(s['quantity'] * s['price']) - s['discount_amount'].fillna(0))
       .merge(p[['product_id','product_name','category','profit_margin']],
              on='product_id', how='left'))

# Revenue by category
# .isin()    → กรองเฉพาะ category ที่ต้องการ
# .reindex() → จัดลำดับ
cr = (tq[tq['category'].isin(['Pastries','Bread','Tarte'])]
        .groupby('category')['revenue'].sum()
        .reindex(['Pastries','Bread','Tarte']))

# Top 3 products by quantity
# .nlargest()  → เลือก N แถวที่มีค่ามากที่สุด
# .rename()    → เปลี่ยนชื่อคอลัมน์
t3 = (tq.groupby('product_name')[['quantity','revenue']].sum()
        .nlargest(3, 'quantity').reset_index()
        .rename(columns={'product_name':'Product Name',
                         'quantity':    'Total Qty',
                         'revenue':     'Total Revenue'}))
t3['Total Revenue'] = t3['Total Revenue'].apply(lambda v: f'${v:,.2f}')

print('Revenue by category:')
print(cr)
print('\nTop 3 products:')
display(t3)

Revenue by category:
category
Pastries    55469.37
Bread          45.81
Tarte       61328.50
Name: revenue, dtype: float64

Top 3 products:


,Product Name,Total Qty,Total Revenue
0,cannelé bordelais,11965,"$55,457.02"
1,pain de campagne,7064,"$32,487.58"
2,macaron pistache,6801,"$31,045.62"


In [4]:
# --- สร้าง PDF ---

with PdfPages('Session1_ProductPerformance.pdf') as pdf:

    # หน้า 1: Bar Chart
    fig, ax = plt.subplots(figsize=(8, 5))
    # .plot(kind='bar') → วาด bar chart จาก Series
    # rot=0 → ไม่หมุน label แกน x
    cr.plot(kind='bar', color=['tomato','steelblue','seagreen'], ax=ax, rot=0)
    ax.set_title('Total Revenue by Product Category', fontsize=14, fontweight='bold', pad=12)
    ax.set_ylabel('Total Revenue ($)'); ax.set_xlabel('Category')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v:,.0f}'))
    ax.grid(axis='y', linestyle='--', alpha=0.5)
    fig.tight_layout()
    pdf.savefig(fig, bbox_inches='tight'); plt.close()

    # หน้า 2: Top 3 Table
    fig, ax = plt.subplots(figsize=(9, 2.5))
    ax.axis('off')
    ax.set_title('Top 3 Best-Selling Products', fontsize=13, fontweight='bold', pad=16)
    tbl = ax.table(cellText=t3.values, colLabels=t3.columns, loc='center', cellLoc='center')
    tbl.auto_set_font_size(False); tbl.set_fontsize(11); tbl.scale(1.5, 2.2)
    # r==0 คือแถว header → ใส่สีพื้นหลัง
    for (r, _), cell in tbl.get_celld().items():
        if r == 0: cell.set_facecolor('tomato'); cell.set_text_props(color='white', fontweight='bold')
        cell.set_edgecolor('lightgray')
    fig.tight_layout()
    pdf.savefig(fig, bbox_inches='tight'); plt.close()

print('✅ Saved Session1_ProductPerformance.pdf')

# === จุดสังเกต ===
# ✔ PDF มี 2 หน้า
# ✔ Bar chart มี 3 category: Pastries, Bread, Tarte
# ✔ Top 3 table มี 3 แถว

✅ Saved Session1_ProductPerformance.pdf
